# Qwen3-VL-8B-Instruct FP16 on Kaggle T4×2 GPU — Production Showcase

This notebook demonstrates the **production-style usage layer** of the repository. It is intentionally separate from the already-published acceptance notebook that qualifies the FP16/T4×2 baseline.

**What this notebook proves**
- one reusable FastAPI server process;
- native `torch.float16` model placement across **Tesla T4 ×2**;
- no CPU or disk weight offload;
- specialized text, vision and schema-driven extraction endpoints;
- multipart image upload only;
- deep multimodal examples with machine-readable evidence;
- optional Cloudflare exposure remains **disabled by default**.


## 1. Attach the Kaggle Model before running

In the notebook sidebar choose **Add Input → Models** and attach:

`dangkhoa2016/qwen-qwen3-vl-8b-instruct`

Expected local model path:

`/kaggle/input/models/dangkhoa2016/qwen-qwen3-vl-8b-instruct/transformers/default/1`

Select **GPU T4 ×2** as the accelerator. This notebook does not silently download a second full checkpoint from Hugging Face.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time, json

REPO_NAME = "Qwen3-VL-8B-Instruct-FP16-On-Kaggle-T4x2-GPU"
MODEL_PATH = Path("/kaggle/input/models/dangkhoa2016/qwen-qwen3-vl-8b-instruct/transformers/default/1")

# When executed from a checked-out repository, use it directly. For a standalone
# public Kaggle notebook, clone the public repository once if it is absent.
repo_candidates = [Path.cwd(), Path("/kaggle/working") / REPO_NAME]
REPO_ROOT = next((p for p in repo_candidates if (p / "src" / "qwen3vl_t4x2").is_dir()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path("/kaggle/working") / REPO_NAME
    subprocess.run([
        "git", "clone", "--depth", "1",
        f"https://github.com/dangkhoa2016/{REPO_NAME}.git",
        str(REPO_ROOT),
    ], check=True)

assert MODEL_PATH.is_dir(), f"Attach the Kaggle Model first: missing {MODEL_PATH}"
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print("REPO_ROOT=", REPO_ROOT)
print("MODEL_PATH=", MODEL_PATH)


## 2. Hardware and runtime snapshot

This is an early fail-fast gate. The showcase should not proceed on a single GPU or a non-T4 accelerator.


In [ ]:
import torch, transformers, accelerate, fastapi, uvicorn

assert torch.cuda.is_available(), "CUDA is required"
assert torch.cuda.device_count() == 2, f"Expected exactly 2 GPUs, got {torch.cuda.device_count()}"
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
assert all("T4" in name for name in gpu_names), f"Expected Tesla T4 ×2, got {gpu_names}"

snapshot = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "fastapi": fastapi.__version__,
    "uvicorn": uvicorn.__version__,
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": gpu_names,
}
print(json.dumps(snapshot, indent=2))


## 3. Optional configuration

The default path is localhost-only. Keep the public tunnel disabled unless you explicitly want external API access.


In [ ]:
os.environ.setdefault("QWEN3VL_MODEL_PATH", str(MODEL_PATH))
os.environ.setdefault("QWEN3VL_PORT", "8000")
os.environ.setdefault("ENABLE_CLOUDFLARE_TUNNEL", "False")

print("ENABLE_CLOUDFLARE_TUNNEL=", os.environ["ENABLE_CLOUDFLARE_TUNNEL"])
print("QWEN3VL_PORT=", os.environ["QWEN3VL_PORT"])
# Never print QWEN3VL_API_KEY.


## 4. Verify the static asset pack

The default showcase uses repository-controlled benchmark assets. External/optional assets are not required for the core run.


In [ ]:
subprocess.run([sys.executable, "scripts/verify_asset_pack.py", "."], check=True)


## 5. Start the REST server and wait for readiness

The server loads the model **once**. All following notebook cells interact through HTTP rather than calling the model directly.


In [ ]:
import signal
from pathlib import Path
import httpx

PORT = int(os.environ["QWEN3VL_PORT"])
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = REPO_ROOT / "server.log"

# Clean up only a server process started by this notebook session.
server_proc = subprocess.Popen(
    [sys.executable, "scripts/start_server.py"],
    stdout=LOG_PATH.open("w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print("SERVER_PID=", server_proc.pid)

ready = False
with httpx.Client(timeout=2.0) as client:
    for _ in range(180):
        if server_proc.poll() is not None:
            raise RuntimeError(LOG_PATH.read_text(errors="replace")[-5000:])
        try:
            response = client.get(f"{BASE_URL}/health")
            if response.status_code == 200 and response.json().get("ready") is True:
                ready = True
                break
        except httpx.HTTPError:
            pass
        time.sleep(1)
assert ready, "REST server did not become ready"
print("SERVER_READY=PASS")


## 6. Runtime identity — baseline proof

`/v1/model` is the reviewer-facing runtime identity. It must report FP16, both T4 GPUs, and no CPU/disk weight offload.


In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    health = client.get("/health").json()
    model_info = client.get("/v1/model").json()

assert health == {"status": "ok", "model_loaded": True, "ready": True}
assert model_info["dtype"] == "torch.float16"
assert model_info["gpu_count"] == 2
assert model_info["gpu_names"] == ["Tesla T4", "Tesla T4"]
assert model_info["cpu_weight_offload"] is False
assert model_info["disk_weight_offload"] is False
print(json.dumps(model_info, indent=2))
print("RUNTIME_IDENTITY_GATE=PASS")


## 7. Text-only API smoke test

Text generation is present as a simple service sanity check; the remainder of the notebook focuses on multimodal capabilities.


In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=120.0) as client:
    r = client.post("/v1/generate", json={
        "prompt": "Reply with exactly: QWEN3VL_T4X2_OK",
        "max_new_tokens": 32,
        "temperature": 0.0,
    })
    r.raise_for_status()
    text_result = r.json()
print(json.dumps(text_result, indent=2, ensure_ascii=False))
assert "QWEN3VL_T4X2_OK" in text_result["output"]
print("TEXT_API_GATE=PASS")


## 8. General Vision Understanding

Run several scene-understanding cases that require object recognition, spatial relationships and multi-object reasoning.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--group", "general_vision",
    "--max-cases", "3",
    "--output-dir", "results/general-vision",
], check=True)


## 9. OCR + Document Understanding

These cases go beyond transcription: the prompts ask about document type, field relationships, operational meaning and arithmetic.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--group", "ocr_document",
    "--max-cases", "4",
    "--output-dir", "results/ocr-document",
], check=True)


## 10. Generic Schema-Driven Structured Extraction

The extraction endpoint is not hard-coded to invoices. The same API accepts different compact schemas and validates model JSON against each normalized schema.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--group", "structured_extraction",
    "--max-cases", "4",
    "--output-dir", "results/structured-extraction",
], check=True)


## 11. Tables, Charts and Visual Reasoning

These cases test lookup, comparison, trend and cross-cell/cross-series reasoning rather than OCR alone.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--group", "tables_charts",
    "--max-cases", "5",
    "--output-dir", "results/tables-charts",
], check=True)


## 12. Multilingual & Mixed-Context Vision

English and Vietnamese prompts/content are mixed deliberately to test cross-language instruction following while preserving names, dates and numbers.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--group", "multilingual",
    "--max-cases", "3",
    "--output-dir", "results/multilingual",
], check=True)


## 13. Consolidated showcase evidence

Run the default case catalog into the canonical evidence files. Free-form vision cases remain manual semantic review; structured extraction cases receive automatic expected-field checks.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_showcase.py",
    "--base-url", BASE_URL,
    "--output-dir", "results",
], check=True)
subprocess.run([sys.executable, "scripts/verify_evidence.py"], check=True)


## 14. Service-level sequential stability

This is a service-layer check, not a replacement for the separate acceptance notebook's stability campaign. It verifies repeated HTTP requests reuse the same loaded server/model and records GPU memory snapshots.


In [ ]:
stability_script = REPO_ROOT / "scripts" / "run_stability.py"
if stability_script.is_file():
    subprocess.run([
        sys.executable, str(stability_script),
        "--base-url", BASE_URL,
        "--requests", "12",
    ], check=True)
else:
    print("STABILITY_SCRIPT_NOT_PRESENT — run after repository implementation is finalized")


## 15. Metrics and final summary

Runtime success, automatic structured validation, and manual free-form quality review remain separate. A successful HTTP status is never treated as semantic proof by itself.


In [ ]:
summary = json.loads((REPO_ROOT / "results" / "showcase-summary.json").read_text())
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    metrics = client.get("/v1/metrics").json()

final = {
    "server_health": summary["server_health"],
    "showcase_cases_total": summary["showcase_cases_total"],
    "showcase_runtime_pass": summary["showcase_runtime_pass"],
    "showcase_runtime_fail": summary["showcase_runtime_fail"],
    "structured_parse_rate": summary["structured_parse_rate"],
    "structured_schema_valid_rate": summary["structured_schema_valid_rate"],
    "semantic_auto_evaluated": summary["semantic_auto_evaluated"],
    "semantic_auto_pass": summary["semantic_auto_pass"],
    "metrics": metrics,
    "cloudflare_enabled": os.environ.get("ENABLE_CLOUDFLARE_TUNNEL", "False"),
    "evidence_jsonl": str(REPO_ROOT / "results" / "showcase-results.jsonl"),
    "evidence_summary": str(REPO_ROOT / "results" / "showcase-summary.json"),
}
print(json.dumps(final, indent=2, ensure_ascii=False))

assert final["showcase_runtime_fail"] == 0
assert final["structured_parse_rate"] == 1.0
assert final["structured_schema_valid_rate"] == 1.0
print("REST_API_GATE=PASS")
print("FINAL_SHOWCASE_RUNTIME=PASS")
print("NOTE: Free-form semantic quality still requires reviewer inspection.")


## 16. Optional authenticated Cloudflare Quick Tunnel

The notebook is complete without public access. Only enable this deliberately:

```python
os.environ["ENABLE_CLOUDFLARE_TUNNEL"] = "True"
os.environ["QWEN3VL_API_KEY"] = "<secret>"  # never print this value
```

Then launch `scripts/start_cloudflare_tunnel.sh`. The script refuses to open the tunnel if the API key is absent. Public requests require `Authorization: Bearer <API_KEY>`, while localhost remains unauthenticated.


## 17. Cleanup helper

Use this only when you are finished with the interactive Kaggle session.


In [ ]:
def stop_showcase_server():
    global server_proc
    if server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait()
    print("SERVER_STOPPED")

print("Cleanup helper ready: call stop_showcase_server() when finished.")
